# Robust04 — Self-Contained Notebook to Reproduce `run_1.res`, `run_2.res`, `run_3.res`

This notebook is **fully self-contained**:

- It does **not** call any external repo scripts (e.g., `generate_runs.py`, `generate_hyde.py`).
- It **can** use cached artifacts (`hyde_all_hypothetical_docs.jsonl`, disk cache) to speed things up.
- If cached artifacts do **not** exist, it can regenerate them (including HyDE generation).

## What this notebook produces

- `run_1.res`
- `run_2.res`
- `run_3.res`

All in standard **6-column TREC format** for the **199 test queries** (qids 351–700 with gaps, i.e. the last 199 lines in `queriesROBUST.txt`).

## Important notes

- HyDE generation uses **Zephyr 7B** and may require a GPU (and/or 4-bit quantization). If you already have `hyde_all_hypothetical_docs.jsonl`, generation will be skipped.
- MonoT5 passage reranking uses **`cramraj8/duqgen-monot5-3b-robust04-1k`** and is compute-heavy.
- The notebook uses a disk cache under `CACHE_DIR` to reuse:
  - document texts fetched from Lucene
  - retrieval baselines
  - MonoT5 passage raw scores


## Presentation agenda

1. **Problem + dataset** (ROBUST04) and the train/test protocol (50 judged + 199 test)
2. **Evaluation metric** (MAP) and what it measures
3. **Pipeline overview** (retrieval → fusion → reranking)
4. **Run 1**: fused sparse+dense retrieval (baseline)
5. **Run 2**: alternative fusion weights/components (baseline)
6. **Run 3**: Run 1 + MonoT5 passage reranking (final submission system)
7. **Reproducibility notes** (caching, randomness, hardware)
8. **References**

## One-slide summary (what differs between the runs)

- **Run 1 (`run_1.res`)**
  - Candidate set: min-max weighted fusion of
    - RM3 (BM25+RM3)
    - SPLADE++
    - SPLADE-v3
    - Dense BGE (query = `orig_hyde`)
- **Run 2 (`run_2.res`)**
  - Candidate set: min-max weighted fusion of
    - RM3 (BM25+RM3)
    - SPLADE++
    - Dense BGE (query = `orig_hyde`)
- **Run 3 (`run_3.res`)**
  - Start from **Run 1’s** candidate set
  - Rerank top `MONOT5P_TOP_N` with **MonoT5-3B passage scoring**, then blend with baseline scores


## Dataset and evaluation protocol (ROBUST04)

- **Collection**: TREC Robust04 newswire collection (via Pyserini prebuilt indices)
- **Queries**: `queriesROBUST.txt` contains **249 title queries**
- **Splits used in this project**
  - **Judged / tuning**: first **50** queries (qids 301–350) with relevance judgments in `qrels_50_Queries`
  - **Test / submission**: remaining **199** queries (qids after 350)

### Metric: MAP (Mean Average Precision)

For each query:

- Traverse the ranked list.
- Every time we see a relevant document, add the precision at that rank.
- Divide by the number of relevant documents for that query.

Then average across queries.

In this notebook:

- We **compute MAP on the 50 judged queries** (because that’s where qrels exist).
- The **199 test queries** do not have qrels available to us.


## Pipeline overview (retrieval → fusion → reranking)

### Retrieval components

We use a mix of classic sparse retrieval, learned sparse retrieval, and dense retrieval:

- **RM3 (BM25 + pseudo-relevance feedback)** over the `robust04` Lucene index
- **SPLADE++** (learned sparse expansion) over a prebuilt impact index
- **SPLADE-v3** (a newer SPLADE variant) over a prebuilt impact index
- **Dense BGE** retrieval over a prebuilt HNSW index

### Fusion (score-level)

For fusion runs, each retriever produces a ranked list with its own scoring scale.

- We apply **per-query min-max normalization** to each retriever’s scores.
- We compute a **weighted sum** of the normalized scores.

This gives us a single fused ranking per query.

### Reranking (Run 3)

Run 3 applies a neural reranker on top of the fused candidate set:

- **MonoT5-3B passage scoring** over the top `MONOT5P_TOP_N` documents
- Combine baseline fused score and reranker score using `MONOT5P_ALPHA`


## Run 1 (`run_1.res`): fused sparse+dense baseline

### Goal
Create a strong **first-stage ranking** by combining complementary retrievers.

### What it is
In our final codebase, `run_1` is the **baseline fused3** run:

- RM3 (BM25 + RM3 PRF)
- SPLADE++
- SPLADE-v3
- Dense BGE

### Query sources
- RM3: `orig`
- SPLADE++: `orig`
- SPLADE-v3: `orig`
- Dense BGE: `orig_hyde`

### Fusion rule (per query)
1. Min-max normalize each retriever’s scores to `[0, 1]`
2. Weighted sum with:
   - `W_RUN3 = [0.55, 0.10, 0.15, 0.20]`

### Judged performance
We report **MAP on the 50 judged queries** in the results section below.


## Run 2 (`run_2.res`): lighter fusion baseline

### Goal
Provide a second, meaningfully different run while staying within a fully reproducible fusion framework.

### What it is
`run_2` is the **baseline fused2** run:

- RM3 (BM25 + RM3 PRF)
- SPLADE++
- Dense BGE

This intentionally **drops SPLADE-v3** compared to Run 1.

### Query sources
- RM3: `orig`
- SPLADE++: `orig`
- Dense BGE: `orig_hyde`

### Fusion rule (per query)
1. Min-max normalize each retriever’s scores to `[0, 1]`
2. Weighted sum with:
   - `W_RUN2 = [0.60, 0.25, 0.15]`

### Judged performance
We report **MAP on the 50 judged queries** in the results section below.


## Run 3 (`run_3.res`): Run 1 + MonoT5 passage reranking

### Goal
Improve ranking quality beyond retrieval/fusion by applying a neural reranker.

### What it is
Run 3 starts from **the same candidate set as Run 1** (baseline fused3), then reranks.

### Reranker: MonoT5-3B passage scoring
We use `cramraj8/duqgen-monot5-3b-robust04-1k` in a **passage-level** setup:

- Split each document into overlapping passages (`MONOT5P_PASSAGE_CHARS`, `MONOT5P_STRIDE_CHARS`)
- Score each passage with MonoT5 using the prompt:
  - `Query: {q} Document: {passage} Relevant:`
- Aggregate passage scores to a document score using `MONOT5P_AGG` (default: `max`)

### Score blending
For the top `MONOT5P_TOP_N` documents:

1. Min-max normalize baseline fused scores
2. Min-max normalize MonoT5 passage scores
3. Combine:

`score = MONOT5P_ALPHA * base_norm + (1 - MONOT5P_ALPHA) * monot5_norm`

Documents outside the top reranked set are appended with a decreasing tail score.

### Judged performance
We report **MAP on the 50 judged queries** in the results section below.


In [ ]:
import os
import re
import json
import math
import time
import gzip
import pickle
import hashlib
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict
from typing import Any, Dict, Iterable, List, Optional, Tuple

# Prevent Lucene memory-segment issues in some environments
os.environ.setdefault(
    "JAVA_TOOL_OPTIONS",
    "-Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false",
)

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
)

# IMPORTANT: importing SpladeQueryEncoder from pyserini.encode (public) can trigger optional faiss modules.
# Using the private module avoids that optional import path.
from pyserini.encode._splade import SpladeQueryEncoder
from pyserini.search.lucene import LuceneSearcher, LuceneImpactSearcher, LuceneHnswDenseSearcher

print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE)


## 0. Configuration (paths, caching, and reproducibility)

This section defines *all* configuration knobs in one place.

- We generate runs for the **test split** (199 qids): the last 199 lines of `Files-20260104/queriesROBUST.txt`.
- We will reuse:
  - `hyde_all_hypothetical_docs.jsonl` if it exists and is complete
  - disk cache entries under `CACHE_DIR` if they exist

If any required artifact is missing, the notebook will generate it.


In [ ]:
# Paths
PROJECT_ROOT = Path('.').resolve()
QUERIES_PATH = PROJECT_ROOT / 'Files-20260104' / 'queriesROBUST.txt'
QRELS_JUDGED_PATH = PROJECT_ROOT / 'Files-20260104' / 'qrels_50_Queries'

# Artifacts we may reuse
HYDE_JSONL_PATH = PROJECT_ROOT / 'hyde_all_hypothetical_docs.jsonl'

# Output run files (submission deliverables)
OUT_RUN_1 = PROJECT_ROOT / 'run_1.res'
OUT_RUN_2 = PROJECT_ROOT / 'run_2.res'
OUT_RUN_3 = PROJECT_ROOT / 'run_3.res'

# Disk cache (pick the same default used in the repo experiments)
CACHE_DIR = Path('/workspace/.cache')

# Split / target
QID_SET = 'test'  # 'judged' (50) | 'test' (199) | 'all' (249)
K = 1000

# Retrieval indices and core settings
RM3_INDEX = 'robust04'
BM25_K1 = 0.9
BM25_B = 0.4
RM3_FB_TERMS = 20
RM3_FB_DOCS = 5
RM3_OQW = 0.5

SPLADEPP_INDEX = 'beir-v1.0.0-robust04.splade-pp-ed'
SPLADEPP_MODEL = 'naver/splade-cocondenser-ensembledistil'

SPLADEV3_INDEX = 'beir-v1.0.0-robust04.splade-v3'
SPLADEV3_MODEL = 'naver/splade-v3-distilbert'

DENSE_INDEX = 'beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw'
DENSE_ENCODER = 'BgeBaseEn15'
DENSE_EF_SEARCH = 1000

# Fusion weights (these match the defaults in generate_runs.py)
W_RUN2 = [0.60, 0.25, 0.15]           # RM3, SPLADE++, Dense
W_RUN3 = [0.55, 0.10, 0.15, 0.20]     # RM3, SPLADE++, SPLADE-v3, Dense

# Query-source configuration to match the winning final command:
# --dense-query-source orig_hyde (others are orig)
QUERY_SOURCE_RM3 = 'orig'
QUERY_SOURCE_SPLADEPP = 'orig'
QUERY_SOURCE_SPLADEV3 = 'orig'
QUERY_SOURCE_DENSE = 'orig_hyde'

# HyDE generation configuration (only used if HYDE_JSONL_PATH is missing/incomplete)
HYDE_MODEL_NAME = 'HuggingFaceH4/zephyr-7b-beta'
HYDE_BATCH_SIZE = 2
HYDE_MAX_NEW_TOKENS = 200
HYDE_GREEDY = False
HYDE_TEMPERATURE = 0.7
HYDE_TOP_P = 0.9
HYDE_LOAD_IN_4BIT = True

# MonoT5 passage reranking configuration (winning final settings)
RERANK3_MONOT5_PASSAGES = True
MONOT5P_MODEL = 'cramraj8/duqgen-monot5-3b-robust04-1k'
MONOT5P_TOP_N = 1000
MONOT5P_ALPHA = 0.3
MONOT5P_BATCH_SIZE = 2
MONOT5P_MAX_LENGTH = 512
MONOT5P_DOC_MAX_CHARS = 20000
MONOT5P_PASSAGE_CHARS = 1500
MONOT5P_STRIDE_CHARS = 1200
MONOT5P_MAX_PASSAGES = 15
MONOT5P_SCORE_TOP_N = 1000
MONOT5P_SCORE_MAX_PASSAGES = 15
MONOT5P_AGG = 'max'      # {max, avg_topk, softmax, hybrid}
MONOT5P_AVG_TOPK = 3
MONOT5P_SOFTMAX_TEMP = 1.0
MONOT5P_HYBRID_LAMBDA = 0.5
MONOT5P_FP16 = True

# Reproduction controls
FORCE_REGEN_HYDE = False
FORCE_REGEN_RUNS = False
FORCE_CACHE_REFRESH = False  # ignores disk cache if True

print('PROJECT_ROOT:', PROJECT_ROOT)
print('QID_SET:', QID_SET)
print('CACHE_DIR:', CACHE_DIR)
print('HYDE_JSONL_PATH:', HYDE_JSONL_PATH)
print('Outputs:', OUT_RUN_1.name, OUT_RUN_2.name, OUT_RUN_3.name)


## 1. Minimal disk cache (self-contained)

We implement a small on-disk cache compatible with our needs:

- Key is a Python object (dict/list/str/etc.) hashed deterministically.
- Value is pickled and gzip-compressed.

This cache is used to speed up:

- fetching Robust04 document texts from Lucene
- baseline retrieval results per query
- MonoT5 passage-level raw scores per query


In [ ]:
def _update_hash(h: "hashlib._Hash", obj: Any) -> None:
    if obj is None:
        h.update(b"n")
        return
    if isinstance(obj, bool):
        h.update(b"b1" if obj else b"b0")
        return
    if isinstance(obj, int):
        h.update(b"i")
        h.update(str(obj).encode("utf-8"))
        h.update(b";")
        return
    if isinstance(obj, float):
        h.update(b"f")
        h.update(repr(obj).encode("utf-8"))
        h.update(b";")
        return
    if isinstance(obj, str):
        h.update(b"s")
        h.update(obj.encode("utf-8"))
        h.update(b";")
        return
    if isinstance(obj, bytes):
        h.update(b"y")
        h.update(obj)
        h.update(b";")
        return
    if isinstance(obj, (list, tuple)):
        h.update(b"[")
        for x in obj:
            _update_hash(h, x)
            h.update(b",")
        h.update(b"]")
        return
    if isinstance(obj, dict):
        h.update(b"{")
        for k in sorted(obj.keys(), key=lambda x: (str(type(x)), repr(x))):
            _update_hash(h, k)
            h.update(b":")
            _update_hash(h, obj[k])
            h.update(b",")
        h.update(b"}")
        return

    h.update(b"r")
    h.update(repr(obj).encode("utf-8"))
    h.update(b";")


def make_hash(obj: Any) -> str:
    h = hashlib.sha256()
    _update_hash(h, obj)
    return h.hexdigest()


@dataclass
class DiskCache:
    cache_dir: Path
    enabled: bool = True
    refresh: bool = False

    def _path(self, namespace: str, key: str) -> Path:
        subdir = self.cache_dir / namespace / key[:2] / key[2:4]
        return subdir / f"{key}.pkl.gz"

    def get(self, namespace: str, key_obj: Any) -> Optional[Any]:
        if (not self.enabled) or self.refresh:
            return None
        key = make_hash(key_obj)
        path = self._path(namespace, key)
        if not path.exists():
            return None
        try:
            with gzip.open(path, "rb") as f:
                return pickle.load(f)
        except Exception:
            return None

    def set(self, namespace: str, key_obj: Any, value: Any) -> None:
        if not self.enabled:
            return
        key = make_hash(key_obj)
        path = self._path(namespace, key)
        path.parent.mkdir(parents=True, exist_ok=True)
        tmp = path.with_name(path.name + ".tmp")
        with gzip.open(tmp, "wb") as f:
            pickle.dump(value, f, protocol=pickle.HIGHEST_PROTOCOL)
        os.replace(tmp, path)


disk_cache = DiskCache(cache_dir=CACHE_DIR, enabled=True, refresh=bool(FORCE_CACHE_REFRESH))
print('disk_cache_dir:', disk_cache.cache_dir)
print('disk_cache_refresh:', disk_cache.refresh)


## 2. Load queries and define judged/test splits

The course setup uses a fixed split:

- **Judged / tuning:** first 50 qids in the file (qids 301–350)
- **Test / submission:** remaining 199 qids

We load the TSV file into an ordered mapping `qid -> query` and then select the target qids depending on `QID_SET`.


In [ ]:
def read_queries_tsv(path: Path) -> Dict[str, str]:
    queries: Dict[str, str] = {}
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            qid, query = line.split('\t', 1)
            queries[str(qid)] = str(query)
    return queries


all_queries = read_queries_tsv(QUERIES_PATH)
all_qids = list(all_queries.keys())
train_qids = all_qids[:50]
test_qids = all_qids[50:]

if QID_SET == 'judged':
    target_qids = train_qids
elif QID_SET == 'all':
    target_qids = all_qids
else:
    target_qids = test_qids

target_queries = {qid: all_queries[qid] for qid in target_qids}

print('total_queries:', len(all_queries))
print('train_qids:', len(train_qids))
print('test_qids:', len(test_qids))
print('target_qids:', len(target_qids))
print('target_qid_first_last:', target_qids[0], target_qids[-1])


## 3. HyDE: load or generate `hyde_all_hypothetical_docs.jsonl`

### What is HyDE used for in this final pipeline?

HyDE provides an additional **hypothetical news passage** for each query. In our winning configuration we use HyDE only for **dense retrieval query text**:

- RM3 query: `orig`
- SPLADE++ query: `orig`
- SPLADE-v3 query: `orig`
- Dense query: `orig_hyde`  (concatenate original query + HyDE passage)

So HyDE affects only the dense retriever component of the fused candidate set.

### Reproducibility policy

- If `hyde_all_hypothetical_docs.jsonl` exists and contains all 249 qids, we will reuse it.
- If it is missing or incomplete, we will generate missing qids with Zephyr 7B.

Because generation uses sampling by default, the *exact* HyDE text may vary across environments. To improve reproducibility, we set a fixed seed before generation.


In [ ]:
def _clean_hyp_text(text: str) -> str:
    s = (text or "").strip()
    if not s:
        return ""
    low = s.lower()
    if "explanation:" in low:
        idx = low.index("explanation:")
        s = s[:idx].strip()
    for prefix in ["title:", "passage:"]:
        if s.lower().startswith(prefix):
            s = s[len(prefix):].strip()
    return s


def load_qid_text_jsonl(path: Path) -> Dict[str, str]:
    out: Dict[str, str] = {}
    if not path.exists():
        return out
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except Exception:
                continue
            qid = str(rec.get('qid', '')).strip()
            txt = _clean_hyp_text(str(rec.get('text', '')))
            if qid and txt:
                out[qid] = txt
    return out


def _ensure_hyde_jsonl_exists(
    *,
    queries: Dict[str, str],
    out_path: Path,
    force_regen: bool,
) -> Dict[str, str]:
    # Load existing (if any)
    existing = {} if force_regen else load_qid_text_jsonl(out_path)
    existing_qids = set(existing.keys())

    missing = [(qid, queries[qid]) for qid in queries.keys() if qid not in existing_qids]

    if (not missing) and (len(existing) == len(queries)):
        print(f"HyDE JSONL OK: {out_path} has {len(existing)}/{len(queries)} qids")
        return existing

    if force_regen and out_path.exists():
        # Rebuild file from scratch
        out_path.unlink()
        existing = {}
        existing_qids = set()
        missing = [(qid, queries[qid]) for qid in queries.keys()]

    print(f"HyDE JSONL needs generation: have {len(existing_qids)}/{len(queries)}; missing {len(missing)}")

    # Fixed seed for best-effort reproducibility
    torch.manual_seed(0)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(0)

    print('Loading HyDE model:', HYDE_MODEL_NAME)
    tok = AutoTokenizer.from_pretrained(HYDE_MODEL_NAME)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = 'left'

    # Model load (optionally 4-bit)
    if HYDE_LOAD_IN_4BIT:
        try:
            from transformers import BitsAndBytesConfig

            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type='nf4',
                bnb_4bit_compute_dtype=torch.float16,
            )
            model = AutoModelForCausalLM.from_pretrained(
                HYDE_MODEL_NAME,
                quantization_config=bnb_config,
                device_map='auto',
            )
        except Exception as e:
            print('4-bit load failed, falling back to float16:', repr(e))
            model = AutoModelForCausalLM.from_pretrained(
                HYDE_MODEL_NAME,
                torch_dtype=torch.float16,
                device_map='auto',
            )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            HYDE_MODEL_NAME,
            torch_dtype=torch.float16,
            device_map='auto',
        )

    model.eval()

    bs = max(1, int(HYDE_BATCH_SIZE))
    gen_kwargs = {
        'max_new_tokens': int(HYDE_MAX_NEW_TOKENS),
        'do_sample': (not HYDE_GREEDY),
        'pad_token_id': int(tok.pad_token_id),
    }
    if not HYDE_GREEDY:
        gen_kwargs.update({'temperature': float(HYDE_TEMPERATURE), 'top_p': float(HYDE_TOP_P)})

    out_path.parent.mkdir(parents=True, exist_ok=True)

    # Append mode allows resume; if we force-regenerated we already deleted the file.
    with out_path.open('a', encoding='utf-8') as f_out:
        for start in range(0, len(missing), bs):
            batch = missing[start:start+bs]
            qids = [x[0] for x in batch]

            batch_prompts = []
            for _, q in batch:
                batch_prompts.append(
                    [
                        {
                            'role': 'system',
                            'content': 'You are a helpful assistant. Write a short news passage that answers the given query.',
                        },
                        {'role': 'user', 'content': f"Query: {q}\nPassage:"},
                    ]
                )

            prompt_texts = [tok.apply_chat_template(p, tokenize=False, add_generation_prompt=True) for p in batch_prompts]
            enc = tok(prompt_texts, return_tensors='pt', padding=True)
            input_len = int(enc['input_ids'].shape[1])
            enc = enc.to(model.device)

            with torch.inference_mode():
                outputs = model.generate(**enc, **gen_kwargs)

            for i, qid in enumerate(qids):
                gen_ids = outputs[i][input_len:].tolist()
                gen_text = tok.decode(gen_ids, skip_special_tokens=True).strip()
                rec = {'qid': str(qid), 'text': gen_text}
                f_out.write(json.dumps(rec) + '\n')
                existing[str(qid)] = _clean_hyp_text(gen_text)

            f_out.flush()

            if (start // bs + 1) % 10 == 0:
                done = min(start + bs, len(missing))
                print(f"HyDE generated {done}/{len(missing)} missing qids")

    # Final load ensures we return a normalized mapping even if generation produced empty strings.
    hyde_docs = load_qid_text_jsonl(out_path)
    if len(hyde_docs) != len(queries):
        missing2 = sorted(set(queries.keys()) - set(hyde_docs.keys()))
        raise RuntimeError(f"HyDE JSONL incomplete after generation: missing {len(missing2)} qids (e.g. {missing2[:10]})")

    print(f"HyDE JSONL ready: {out_path} has {len(hyde_docs)}/{len(queries)} qids")
    return hyde_docs


# Generate/load HyDE for *all* queries, because query sources can be applied across qid sets.
hyde_docs_all = _ensure_hyde_jsonl_exists(
    queries=all_queries,
    out_path=HYDE_JSONL_PATH,
    force_regen=bool(FORCE_REGEN_HYDE),
)


## 4. Query-source resolution (Orig vs HyDE vs Orig+HyDE)

This pipeline supports multiple query text sources:

- `orig`: original title query
- `hyde`: HyDE hypothetical document
- `orig_hyde`: concatenate `orig + ' ' + hyde`

In the final configuration used to generate the submission runs:

- RM3 uses `orig`
- SPLADE++ uses `orig`
- SPLADE-v3 uses `orig`
- Dense uses `orig_hyde`


In [ ]:
def resolve_query_source(
    *,
    qid: str,
    orig_query: str,
    source: str,
    hyde_docs: Dict[str, str],
) -> str:
    src = str(source)
    if src == 'orig':
        return orig_query
    if src == 'hyde':
        return hyde_docs.get(qid, orig_query)
    if src == 'orig_hyde':
        hyp = hyde_docs.get(qid, '')
        return orig_query if not hyp else (orig_query + ' ' + hyp)
    return orig_query


# quick smoke-check
qid0 = target_qids[0]
print('example qid:', qid0)
print('orig:', all_queries[qid0])
print('dense_query_source:', QUERY_SOURCE_DENSE)
print('dense_query_text_prefix:', resolve_query_source(qid=qid0, orig_query=all_queries[qid0], source=QUERY_SOURCE_DENSE, hyde_docs=hyde_docs_all)[:120])


## 5. Retrieval + fusion helpers (self-contained)

This section contains the exact mechanics used to build the three runs:

- **Retrieval**: query each searcher and collect `(docid, score)`
- **Min-max normalization**: normalize each retriever’s scores per query into `[0, 1]`
- **Weighted fusion**: sum normalized scores with fixed weights
- **`ensure_k`**: if fusion produces < `K` unique docs, backfill from an RM3 fallback list

These functions are written to match the behavior of our run-generation code.


In [ ]:
@dataclass
class SearchArtifacts:
    docids_scores: Dict[str, float]
    ranked: List[Tuple[str, float]]


def retrieve(searcher, query: str, k: int) -> SearchArtifacts:
    hits = searcher.search(query, k=k)
    ranked = [(h.docid, float(h.score)) for h in hits]
    scores = {docid: score for docid, score in ranked}
    return SearchArtifacts(docids_scores=scores, ranked=ranked)


def minmax_norm(scores_dict: Dict[str, float]) -> Dict[str, float]:
    if not scores_dict:
        return {}
    vals = list(scores_dict.values())
    mn, mx = min(vals), max(vals)
    if mx - mn < 1e-9:
        return {d: 0.0 for d in scores_dict}
    return {d: (float(s) - float(mn)) / (float(mx) - float(mn)) for d, s in scores_dict.items()}


def fuse_weighted_minmax(
    runs_scores: List[Dict[str, float]],
    weights: List[float],
    depth: int,
) -> List[Tuple[str, float]]:
    norms = [minmax_norm(rs) for rs in runs_scores]
    docs = set()
    for n in norms:
        docs |= set(n.keys())

    fused_scores: Dict[str, float] = {}
    for d in docs:
        s = 0.0
        for w, n in zip(weights, norms):
            s += float(w) * float(n.get(d, 0.0))
        fused_scores[d] = float(s)

    ranked = sorted(fused_scores.items(), key=lambda x: (-x[1], x[0]))
    return ranked[: int(depth)]


def ensure_k(
    ranked: List[Tuple[str, float]],
    fallback: List[Tuple[str, float]],
    k: int,
) -> List[Tuple[str, float]]:
    if len(ranked) >= int(k):
        return ranked[: int(k)]

    seen = {d for d, _ in ranked}
    out = list(ranked)
    for d, s in fallback:
        if d in seen:
            continue
        out.append((d, float(s)))
        seen.add(d)
        if len(out) >= int(k):
            break
    return out


def write_trec_run(path: Path, run: Dict[str, List[Tuple[str, float]]], tag: str) -> None:
    with path.open('w', encoding='utf-8') as f:
        for qid in sorted(run.keys(), key=int):
            ranked = run[qid]
            for rank, (docid, score) in enumerate(ranked, start=1):
                f.write(f"{qid} Q0 {docid} {rank} {float(score):.6f} {tag}\n")


def chunked(items: List[str], batch_size: int) -> Iterable[List[str]]:
    bs = int(batch_size)
    if bs <= 0:
        raise ValueError('batch_size must be > 0')
    for i in range(0, len(items), bs):
        yield items[i:i+bs]


## 6. Fetch Robust04 document text (for reranking)

MonoT5 reranking needs document text.

- Our retrieval indexes (SPLADE/dense) do not necessarily expose full raw text.
- We use the `robust04` Lucene index (via `LuceneSearcher.doc(docid).raw()`) to fetch raw HTML.
- We strip tags and collapse whitespace to get plain text.

We apply `DOC_MAX_CHARS` to avoid extremely long inputs.

We cache:

- in-memory per-notebook (`mem_cache`)
- on disk (`disk_cache`, namespace `doc_texts`) keyed by `(docid, max_chars)`


In [ ]:
_TAG_RE = re.compile(r"<[^>]+>")
_WS_RE = re.compile(r"\s+")


def raw_to_text(raw: str) -> str:
    s = _TAG_RE.sub(" ", raw or "")
    s = _WS_RE.sub(" ", s)
    return s.strip()


def fetch_doc_texts_disk_cached(
    searcher: LuceneSearcher,
    docids: List[str],
    mem_cache: Dict[str, str],
    max_chars: int,
    disk_cache: DiskCache,
) -> List[str]:
    out: List[str] = []
    for docid in docids:
        if docid in mem_cache:
            out.append(mem_cache[docid])
            continue

        key = {"index": RM3_INDEX, "docid": str(docid), "max_chars": int(max_chars)}
        txt = disk_cache.get("doc_texts", key)
        if txt is None:
            try:
                doc = searcher.doc(str(docid))
                raw = "" if doc is None else (doc.raw() or "")
            except Exception:
                raw = ""
            txt = raw_to_text(raw)
            if int(max_chars) > 0:
                txt = txt[: int(max_chars)]
            disk_cache.set("doc_texts", key, txt)

        mem_cache[str(docid)] = str(txt)
        out.append(str(txt))

    return out


## 7. MonoT5 passage-level scoring (self-contained)

We replicate the exact scoring strategy:

- Input format: `Query: {q} Document: {passage} Relevant:`
- Get decoder first-step logits for tokens `true` and `false`
- Score = `logit(true) - logit(false)`

For passage reranking:

- Split each document into overlapping passages by character window (`PASSAGE_CHARS`, `STRIDE_CHARS`)
- Score every passage
- Keep raw per-passage scores (`docid -> [scores...]`) so we can aggregate in multiple ways
- Aggregate raw scores into per-doc scores (`max`, `avg_topk`, `softmax`, `hybrid`)

We cache **raw** passage scores on disk (`namespace=monot5p_raw`) keyed by the query and doc list.


In [ ]:
def _split_passages(text: str, passage_chars: int, stride_chars: int, max_passages: int) -> List[str]:
    t = text or ""
    if int(passage_chars) <= 0:
        return [t]
    stride = int(stride_chars)
    if stride <= 0:
        stride = int(passage_chars)
    mp = int(max_passages)
    if mp <= 0:
        mp = 1

    out: List[str] = []
    i = 0
    while i < len(t) and len(out) < mp:
        seg = t[i:i+int(passage_chars)]
        if seg:
            out.append(seg)
        i += stride
    if not out:
        out = [""]
    return out


def compute_monot5_passage_raw_scores(
    tokenizer: AutoTokenizer,
    model: AutoModelForSeq2SeqLM,
    true_id: int,
    false_id: int,
    query: str,
    docids: List[str],
    doc_texts: List[str],
    device: str,
    batch_size: int,
    max_length: int,
    passage_chars: int,
    stride_chars: int,
    max_passages: int,
) -> Dict[str, List[float]]:
    decoder_start = model.config.decoder_start_token_id
    if decoder_start is None:
        decoder_start = tokenizer.pad_token_id

    ex_docids: List[str] = []
    ex_texts: List[str] = []
    for d, t in zip(docids, doc_texts):
        passages = _split_passages(
            t,
            passage_chars=int(passage_chars),
            stride_chars=int(stride_chars),
            max_passages=int(max_passages),
        )
        for p in passages:
            ex_docids.append(str(d))
            ex_texts.append(f"Query: {query} Document: {p} Relevant:")

    doc_to_scores: Dict[str, List[float]] = defaultdict(list)

    with torch.no_grad():
        for batch_docids, batch_text in zip(chunked(ex_docids, int(batch_size)), chunked(ex_texts, int(batch_size))):
            enc = tokenizer(
                batch_text,
                padding=True,
                truncation=True,
                max_length=int(max_length),
                return_tensors='pt',
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            decoder_input_ids = torch.full(
                (len(batch_text), 1),
                int(decoder_start),
                dtype=torch.long,
                device=device,
            )
            logits = model(**enc, decoder_input_ids=decoder_input_ids).logits
            step = logits[:, 0, :]
            batch_scores = (step[:, true_id] - step[:, false_id]).detach().cpu().tolist()
            for d, s in zip(batch_docids, batch_scores):
                doc_to_scores[str(d)].append(float(s))

    return {str(d): (doc_to_scores.get(str(d), []) or []) for d in docids}


def aggregate_monot5_passage_scores(
    raw_scores: Dict[str, List[float]],
    docids: List[str],
    agg: str,
    avg_topk: int,
    max_passages: int,
    softmax_temp: float,
    hybrid_lambda: float,
) -> Dict[str, float]:
    k_passages = max(1, int(max_passages))
    out: Dict[str, float] = {}
    for d in docids:
        scores = (raw_scores.get(str(d), []) or [])[:k_passages]
        if not scores:
            out[str(d)] = 0.0
            continue

        if str(agg) == 'avg_topk':
            k_take = max(1, int(avg_topk))
            topk = sorted(scores, reverse=True)[:k_take]
            out[str(d)] = float(sum(topk) / float(len(topk)))
        elif str(agg) == 'softmax':
            t = float(softmax_temp)
            if not (t > 0.0):
                t = 1.0
            m = float(max(scores)) / t
            exps = [math.exp(float(s) / t - m) for s in scores]
            denom = float(sum(exps))
            if denom <= 0.0:
                out[str(d)] = float(max(scores))
            else:
                out[str(d)] = float(sum(e * float(s) for e, s in zip(exps, scores)) / denom)
        elif str(agg) == 'hybrid':
            lam = float(hybrid_lambda)
            if lam < 0.0:
                lam = 0.0
            if lam > 1.0:
                lam = 1.0
            max_s = float(max(scores))
            k_take = max(1, int(avg_topk))
            topk = sorted(scores, reverse=True)[:k_take]
            avg_s = float(sum(topk) / float(len(topk)))
            out[str(d)] = float(lam * max_s + (1.0 - lam) * avg_s)
        else:
            out[str(d)] = float(max(scores))

    return out


## 8. Initialize searchers (RM3, SPLADE++, SPLADE-v3, Dense)

We initialize four retrieval components:

1. **RM3** over `robust04` (LuceneSearcher with RM3 enabled)
2. **SPLADE++** impact index
3. **SPLADE-v3** impact index
4. **Dense BGE** HNSW index

These are then fused (min-max normalization + weighted sum) to create the candidate sets for `run_2` and `run_3`.


In [ ]:
def init_searchers(device: str):
    rm3 = LuceneSearcher.from_prebuilt_index(RM3_INDEX)
    rm3.set_bm25(float(BM25_K1), float(BM25_B))
    rm3.set_rm3(int(RM3_FB_TERMS), int(RM3_FB_DOCS), float(RM3_OQW))

    bm25 = LuceneSearcher.from_prebuilt_index(RM3_INDEX)
    bm25.set_bm25(float(BM25_K1), float(BM25_B))

    spladepp_encoder = SpladeQueryEncoder(SPLADEPP_MODEL, device=device)
    spladepp = LuceneImpactSearcher.from_prebuilt_index(SPLADEPP_INDEX, spladepp_encoder)

    spladev3_encoder = SpladeQueryEncoder(SPLADEV3_MODEL, device=device)
    spladev3 = LuceneImpactSearcher.from_prebuilt_index(SPLADEV3_INDEX, spladev3_encoder)

    dense = LuceneHnswDenseSearcher.from_prebuilt_index(
        DENSE_INDEX,
        ef_search=int(DENSE_EF_SEARCH),
        encoder=DENSE_ENCODER,
    )

    return rm3, bm25, spladepp, spladev3, dense


t0 = time.time()
rm3, bm25, spladepp, spladev3, dense = init_searchers(DEVICE)
print('searchers initialized in', round(time.time() - t0, 1), 'sec')


## 9. Initialize MonoT5 for passage reranking (run_3)

`run_3` is produced by:

1. Building the **fused run_3 candidate set** (`fused3`) using min-max weighted fusion over:
   - RM3
   - SPLADE++
   - SPLADE-v3
   - Dense BGE (using `orig_hyde` query text)
2. Applying **MonoT5-3B passage-level scoring** to the top `MONOT5P_TOP_N` candidates.
3. Min-max normalizing:
   - the baseline fused scores
   - the MonoT5 passage scores
4. Combining them with `MONOT5P_ALPHA`:

`final = alpha * norm(baseline) + (1 - alpha) * norm(monot5_passage)`

We cache raw passage scores on disk so reruns can be fast.


## Judged-split results (MAP on qids 301–350)

For the class presentation we report the effectiveness of each run on the **50 judged queries**.

- These numbers are **computed by this notebook** from `qrels_50_Queries`.
- They may take time the first time you run them (especially Run 3 with MonoT5 passage reranking), but results are cached.

**Important**: the official leaderboard evaluation is on the 199 test queries, but those qrels are not available. So judged MAP is the metric we can report transparently.


In [ ]:
monot5p_tokenizer = None
monot5p_model = None
true_id = None
false_id = None

if bool(RERANK3_MONOT5_PASSAGES):
    t0 = time.time()
    print('Loading MonoT5 passage model:', MONOT5P_MODEL)
    monot5p_tokenizer = AutoTokenizer.from_pretrained(MONOT5P_MODEL)

    load_kwargs = {}
    if bool(MONOT5P_FP16) and str(DEVICE).startswith('cuda'):
        load_kwargs['torch_dtype'] = torch.float16

    monot5p_model = AutoModelForSeq2SeqLM.from_pretrained(MONOT5P_MODEL, **load_kwargs)
    monot5p_model.to(DEVICE)
    if bool(MONOT5P_FP16) and str(DEVICE).startswith('cuda'):
        monot5p_model.half()
    monot5p_model.eval()

    true_ids = monot5p_tokenizer.encode('true', add_special_tokens=False)
    false_ids = monot5p_tokenizer.encode('false', add_special_tokens=False)
    if (not true_ids) or (not false_ids):
        raise RuntimeError("could not tokenize 'true'/'false'")
    true_id = int(true_ids[0])
    false_id = int(false_ids[0])

    print('MonoT5 init done in', round(time.time() - t0, 1), 'sec')

# in-notebook doc text memoization
_doc_text_cache: Dict[str, str] = {}


## 10. Generate the three runs (with caching)

We reproduce the **exact** run definitions from `generate_runs.py`:

- **`run_1`**: the *baseline* fused-3 run (RM3 + SPLADE++ + SPLADE-v3 + Dense) using `W_RUN3`
- **`run_2`**: the *baseline* fused-2 run (RM3 + SPLADE++ + Dense) using `W_RUN2` (default `--run2-method fusion`)
- **`run_3`**: start from the same baseline fused-3 run as `run_1`, then apply **MonoT5 passage reranking** to the top `MONOT5P_TOP_N`

We cache baseline retrieval triples `(rm3_ranked, fused2, fused3)` under `namespace=generate_runs_baseline`.


## Evaluation utilities (judged MAP)

This section defines the helper functions used to compute **MAP** for each run on the **50 judged queries** (qids 301–350) using `qrels_50_Queries`.

- The actual presentation-friendly output is printed later in **“Judged MAP (presentation output)”**.
- These helpers generate judged runs **in-memory** (they do **not** overwrite the submission `.res` files).


In [ ]:
def read_qrels(path: Path) -> Dict[str, Dict[str, int]]:
    qrels: Dict[str, Dict[str, int]] = defaultdict(dict)
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 4:
                continue
            qid, _, docid, rel_s = parts[:4]
            try:
                rel = int(rel_s)
            except Exception:
                continue
            qrels[str(qid)][str(docid)] = int(rel)
    return dict(qrels)


def average_precision(ranked_docids: List[str], qrel: Dict[str, int]) -> float:
    rel_total = sum(1 for _, r in qrel.items() if int(r) > 0)
    if rel_total <= 0:
        return 0.0

    hits = 0
    s = 0.0
    for i, d in enumerate(ranked_docids, start=1):
        if int(qrel.get(str(d), 0)) > 0:
            hits += 1
            s += float(hits) / float(i)
    return float(s) / float(rel_total)


def mean_ap(run: Dict[str, List[Tuple[str, float]]], qrels: Dict[str, Dict[str, int]]) -> float:
    if not qrels:
        return 0.0
    aps: List[float] = []
    for qid in sorted(qrels.keys(), key=int):
        ranked = run.get(str(qid), [])
        ranked_docids = [d for d, _ in ranked]
        aps.append(average_precision(ranked_docids, qrels[str(qid)]))
    return float(sum(aps) / float(len(aps)))


def evaluate_judged_maps() -> Dict[str, float]:
    if 'generate_all_runs' not in globals():
        raise RuntimeError('generate_all_runs is not defined yet; run the run-generation section first')

    qrels_judged = read_qrels(QRELS_JUDGED_PATH)
    judged_qids = sorted(list(qrels_judged.keys()), key=int)

    _prev_target_qids = target_qids
    try:
        globals()['target_qids'] = [str(q) for q in judged_qids]
        t0 = time.time()
        judged_run_1, judged_run_2, judged_run_3 = generate_all_runs()
        print('judged run generation done in', round(time.time() - t0, 1), 'sec')
    finally:
        globals()['target_qids'] = _prev_target_qids

    return {
        'run_1': mean_ap(judged_run_1, qrels_judged),
        'run_2': mean_ap(judged_run_2, qrels_judged),
        'run_3': mean_ap(judged_run_3, qrels_judged),
    }


In [ ]:
def generate_all_runs() -> Tuple[
    Dict[str, List[Tuple[str, float]]],
    Dict[str, List[Tuple[str, float]]],
    Dict[str, List[Tuple[str, float]]],
]:
    run_1: Dict[str, List[Tuple[str, float]]] = {}
    run_2: Dict[str, List[Tuple[str, float]]] = {}
    run_3: Dict[str, List[Tuple[str, float]]] = {}

    for i, qid in enumerate(target_qids, start=1):
        query = all_queries[qid]

        q_rm3 = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_RM3),
            hyde_docs=hyde_docs_all,
        )
        q_pp = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_SPLADEPP),
            hyde_docs=hyde_docs_all,
        )
        q_v3 = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_SPLADEV3),
            hyde_docs=hyde_docs_all,
        )
        q_dense = resolve_query_source(
            qid=str(qid),
            orig_query=query,
            source=str(QUERY_SOURCE_DENSE),
            hyde_docs=hyde_docs_all,
        )

        baseline_key = {
            "qid": str(qid),
            "query": str(query),
            "query_sources": {
                "rm3": str(QUERY_SOURCE_RM3),
                "spladepp": str(QUERY_SOURCE_SPLADEPP),
                "spladev3": str(QUERY_SOURCE_SPLADEV3),
                "dense": str(QUERY_SOURCE_DENSE),
            },
            "query_texts": {
                "rm3": str(q_rm3),
                "spladepp": str(q_pp),
                "spladev3": str(q_v3),
                "dense": str(q_dense),
            },
            "k": int(K),
            "rm3": {"index": str(RM3_INDEX), "bm25": [float(BM25_K1), float(BM25_B)], "rm3": [int(RM3_FB_TERMS), int(RM3_FB_DOCS), float(RM3_OQW)]},
            "spladepp_index": str(SPLADEPP_INDEX),
            "spladev3_index": str(SPLADEV3_INDEX),
            "dense_index": str(DENSE_INDEX),
            "dense_ef_search": int(DENSE_EF_SEARCH),
            "w_run2": list(W_RUN2),
            "w_run3": list(W_RUN3),
        }

        cached_baseline = disk_cache.get("generate_runs_baseline", baseline_key)
        if cached_baseline is None:
            rm3_art = retrieve(rm3, q_rm3, k=int(K))
            pp_art = retrieve(spladepp, q_pp, k=int(K))
            v3_art = retrieve(spladev3, q_v3, k=int(K))
            dense_art = retrieve(dense, q_dense, k=int(K))

            run1_ranked = rm3_art.ranked[: int(K)]
            fallback_zero = [(d, 0.0) for d, _ in rm3_art.ranked]

            fused2 = fuse_weighted_minmax(
                [rm3_art.docids_scores, pp_art.docids_scores, dense_art.docids_scores],
                list(W_RUN2),
                depth=int(K),
            )
            fused2 = ensure_k(fused2, fallback_zero, k=int(K))

            fused3 = fuse_weighted_minmax(
                [rm3_art.docids_scores, pp_art.docids_scores, v3_art.docids_scores, dense_art.docids_scores],
                list(W_RUN3),
                depth=int(K),
            )
            fused3 = ensure_k(fused3, fallback_zero, k=int(K))

            disk_cache.set("generate_runs_baseline", baseline_key, (run1_ranked, fused2, fused3))
        else:
            run1_ranked, fused2, fused3 = cached_baseline
            fallback_zero = [(d, 0.0) for d, _ in run1_ranked]
            fused2 = ensure_k(list(fused2), fallback_zero, k=int(K))
            fused3 = ensure_k(list(fused3), fallback_zero, k=int(K))

        fused3_base = list(fused3)
        run_1[str(qid)] = fused3_base
        run_2[str(qid)] = list(fused2)

        fused3_for_rerank = list(fused3_base)

        if bool(RERANK3_MONOT5_PASSAGES):
            assert monot5p_tokenizer is not None
            assert monot5p_model is not None
            assert true_id is not None
            assert false_id is not None

            score_top_n = int(MONOT5P_SCORE_TOP_N) if MONOT5P_SCORE_TOP_N is not None else int(MONOT5P_TOP_N)
            score_top_n = max(int(score_top_n), int(MONOT5P_TOP_N))

            score_max_passages = int(MONOT5P_SCORE_MAX_PASSAGES) if MONOT5P_SCORE_MAX_PASSAGES is not None else int(MONOT5P_MAX_PASSAGES)
            score_max_passages = max(int(score_max_passages), int(MONOT5P_MAX_PASSAGES))

            score_pairs = fused3_for_rerank[: int(score_top_n)]
            score_docids = [d for d, _ in score_pairs]

            raw_key = {
                "qid": str(qid),
                "query": str(query),
                "docids": list(score_docids),
                "model_name": str(MONOT5P_MODEL),
                "device": str(DEVICE),
                "batch_size": int(MONOT5P_BATCH_SIZE),
                "max_length": int(MONOT5P_MAX_LENGTH),
                "use_fp16": bool(MONOT5P_FP16),
                "doc_max_chars": int(MONOT5P_DOC_MAX_CHARS),
                "passage_chars": int(MONOT5P_PASSAGE_CHARS),
                "stride_chars": int(MONOT5P_STRIDE_CHARS),
                "max_passages": int(score_max_passages),
            }

            raw_scores = disk_cache.get("monot5p_raw", raw_key)
            if raw_scores is None:
                score_texts = fetch_doc_texts_disk_cached(
                    rm3,
                    score_docids,
                    mem_cache=_doc_text_cache,
                    max_chars=int(MONOT5P_DOC_MAX_CHARS),
                    disk_cache=disk_cache,
                )
                raw_scores = compute_monot5_passage_raw_scores(
                    monot5p_tokenizer,
                    monot5p_model,
                    int(true_id),
                    int(false_id),
                    query=str(query),
                    docids=score_docids,
                    doc_texts=score_texts,
                    device=str(DEVICE),
                    batch_size=int(MONOT5P_BATCH_SIZE),
                    max_length=int(MONOT5P_MAX_LENGTH),
                    passage_chars=int(MONOT5P_PASSAGE_CHARS),
                    stride_chars=int(MONOT5P_STRIDE_CHARS),
                    max_passages=int(score_max_passages),
                )
                disk_cache.set("monot5p_raw", raw_key, raw_scores)

            top_pairs = fused3_for_rerank[: int(MONOT5P_TOP_N)]
            top_docids = [d for d, _ in top_pairs]

            extra_top = aggregate_monot5_passage_scores(
                raw_scores,
                top_docids,
                agg=str(MONOT5P_AGG),
                avg_topk=int(MONOT5P_AVG_TOPK),
                max_passages=int(MONOT5P_MAX_PASSAGES),
                softmax_temp=float(MONOT5P_SOFTMAX_TEMP),
                hybrid_lambda=float(MONOT5P_HYBRID_LAMBDA),
            )

            base_scores = {d: float(s) for d, s in top_pairs}
            base_norm = minmax_norm(base_scores)
            extra_norm = minmax_norm({d: float(extra_top.get(d, 0.0)) for d in top_docids})

            alpha = float(MONOT5P_ALPHA)
            comb = {d: alpha * base_norm.get(d, 0.0) + (1.0 - alpha) * extra_norm.get(d, 0.0) for d in top_docids}
            reranked_top = sorted(comb.items(), key=lambda x: (-x[1], x[0]))

            reranked_set = {d for d, _ in reranked_top}
            tail_docids = [d for d, _ in fused3_for_rerank if d not in reranked_set]

            tail_start = (reranked_top[-1][1] if reranked_top else 0.0) - 1.0
            tail_step = 1e-3
            tail_scores = {d: float(tail_start) - float(tail_step) * i for i, d in enumerate(tail_docids, start=1)}

            fused3_final = [(d, float(comb[d])) for d, _ in reranked_top] + [(d, float(tail_scores[d])) for d in tail_docids]
            fused3_final = fused3_final[: int(K)]
        else:
            fused3_final = fused3_for_rerank[: int(K)]

        run_3[str(qid)] = fused3_final

        if i % 10 == 0:
            print(f"processed {i}/{len(target_qids)} queries")

    return run_1, run_2, run_3


if (not FORCE_REGEN_RUNS) and OUT_RUN_1.exists() and OUT_RUN_2.exists() and OUT_RUN_3.exists():
    print('Run files already exist; skipping regeneration (set FORCE_REGEN_RUNS=True to regenerate).')
else:
    t0 = time.time()
    run_1, run_2, run_3 = generate_all_runs()
    print('Run generation done in', round(time.time() - t0, 1), 'sec')

    write_trec_run(OUT_RUN_1, run_1, tag='run_1')
    write_trec_run(OUT_RUN_2, run_2, tag='run_2')
    write_trec_run(OUT_RUN_3, run_3, tag='run_3')
    print('Wrote:', OUT_RUN_1, OUT_RUN_2, OUT_RUN_3)


## 11. Verify the generated run files

This section validates that each output file:

- Uses valid **6-column TREC format**
- Contains **exactly the target qids** for the chosen `QID_SET`
- Contains **exactly `K` results per query**
- Has ranks `1..K` with **no duplicate docids per query**


In [ ]:
def parse_trec_run_lines(path: Path) -> Dict[str, List[Tuple[int, str, float, str]]]:
    run: Dict[str, List[Tuple[int, str, float, str]]] = defaultdict(list)
    with path.open('r', encoding='utf-8') as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) != 6:
                raise ValueError(f"{path.name}: line {ln} expected 6 columns, got {len(parts)}")
            qid, q0, docid, rank_s, score_s, tag = parts
            if q0 != 'Q0':
                raise ValueError(f"{path.name}: line {ln} col2 expected Q0, got {q0}")
            try:
                rank = int(rank_s)
            except Exception:
                raise ValueError(f"{path.name}: line {ln} rank not int: {rank_s}")
            try:
                score = float(score_s)
            except Exception:
                raise ValueError(f"{path.name}: line {ln} score not float: {score_s}")
            run[str(qid)].append((int(rank), str(docid), float(score), str(tag)))
    return dict(run)


def verify_run_file(path: Path, *, expected_qids: List[str], k: int, expected_tag: Optional[str]) -> None:
    exp = [str(x) for x in expected_qids]
    lines = parse_trec_run_lines(path)

    got = sorted(lines.keys(), key=int)
    if sorted(exp, key=int) != got:
        missing = sorted(set(exp) - set(got), key=int)
        extra = sorted(set(got) - set(exp), key=int)
        raise AssertionError(f"{path.name}: qid mismatch. missing={missing[:10]} extra={extra[:10]}")

    for qid in exp:
        rows = lines[qid]
        if len(rows) != int(k):
            raise AssertionError(f"{path.name}: qid {qid} has {len(rows)} lines, expected {k}")

        ranks = [r for r, _, _, _ in rows]
        if len(set(ranks)) != len(ranks):
            raise AssertionError(f"{path.name}: qid {qid} has duplicate ranks")
        if set(ranks) != set(range(1, int(k) + 1)):
            raise AssertionError(f"{path.name}: qid {qid} ranks are not exactly 1..{k}")

        docids = [d for _, d, _, _ in rows]
        if len(set(docids)) != len(docids):
            raise AssertionError(f"{path.name}: qid {qid} has duplicate docids")

        if expected_tag is not None:
            tags = {t for _, _, _, t in rows}
            if tags != {str(expected_tag)}:
                raise AssertionError(f"{path.name}: qid {qid} has unexpected tags: {sorted(tags)}")


expected_qids = target_qids

print('Verifying:', OUT_RUN_1)
verify_run_file(OUT_RUN_1, expected_qids=expected_qids, k=K, expected_tag='run_1')
print('OK:', OUT_RUN_1.name)

print('Verifying:', OUT_RUN_2)
verify_run_file(OUT_RUN_2, expected_qids=expected_qids, k=K, expected_tag='run_2')
print('OK:', OUT_RUN_2.name)

print('Verifying:', OUT_RUN_3)
verify_run_file(OUT_RUN_3, expected_qids=expected_qids, k=K, expected_tag='run_3')
print('OK:', OUT_RUN_3.name)


## Judged MAP (presentation output)

Run this cell to print the **MAP on the 50 judged queries** for `run_1`, `run_2`, and `run_3`.

- On the first run, this can take time (especially `run_3`) depending on cache/GPU.
- After the first run, the result is cached to disk and prints instantly.


In [ ]:
judged_map_key = {
    'qrels_path': str(QRELS_JUDGED_PATH),
    'rm3': {'index': str(RM3_INDEX), 'bm25': [float(BM25_K1), float(BM25_B)], 'rm3': [int(RM3_FB_TERMS), int(RM3_FB_DOCS), float(RM3_OQW)]},
    'spladepp_index': str(SPLADEPP_INDEX),
    'spladev3_index': str(SPLADEV3_INDEX),
    'dense_index': str(DENSE_INDEX),
    'dense_query_source': str(QUERY_SOURCE_DENSE),
    'w_run2': list(W_RUN2),
    'w_run3': list(W_RUN3),
    'monot5p': {
        'enabled': bool(RERANK3_MONOT5_PASSAGES),
        'model': str(MONOT5P_MODEL),
        'top_n': int(MONOT5P_TOP_N),
        'alpha': float(MONOT5P_ALPHA),
        'batch_size': int(MONOT5P_BATCH_SIZE),
        'max_length': int(MONOT5P_MAX_LENGTH),
        'doc_max_chars': int(MONOT5P_DOC_MAX_CHARS),
        'passage_chars': int(MONOT5P_PASSAGE_CHARS),
        'stride_chars': int(MONOT5P_STRIDE_CHARS),
        'max_passages': int(MONOT5P_MAX_PASSAGES),
        'agg': str(MONOT5P_AGG),
        'avg_topk': int(MONOT5P_AVG_TOPK),
        'softmax_temp': float(MONOT5P_SOFTMAX_TEMP),
        'hybrid_lambda': float(MONOT5P_HYBRID_LAMBDA),
        'fp16': bool(MONOT5P_FP16),
    },
}

cached_maps = disk_cache.get('judged_map', judged_map_key)
if cached_maps is None:
    maps = evaluate_judged_maps()
    disk_cache.set('judged_map', judged_map_key, maps)
else:
    maps = cached_maps

print(f"Judged MAP (qids 301–350)\n  run_1: {maps['run_1']:.4f}\n  run_2: {maps['run_2']:.4f}\n  run_3: {maps['run_3']:.4f}")


## References

- **[ROBUST04 / TREC Robust Track]**
  - Voorhees, E. M. (2004). *Overview of the TREC 2004 Robust Retrieval Track.*
- **[BM25 / Okapi]**
  - Robertson, S., & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.*
- **[RM3 pseudo-relevance feedback]**
  - Lavrenko, V., & Croft, W. B. (2001). *Relevance-Based Language Models.* (RM-style feedback foundations)
- **[SPLADE (learned sparse retrieval)]**
  - Formal, T. et al. (2021). *SPLADE: Sparse Lexical and Expansion Model for First Stage Ranking.*
- **[HyDE (Hypothetical Document Embeddings)]**
  - Gao, L. et al. (2023). *Precise Zero-Shot Dense Retrieval without Relevance Labels.*
- **[BGE dense embeddings]**
  - BAAI (2023). *BGE embedding models* (used via Pyserini prebuilt dense index)
- **[MonoT5 reranking]**
  - Nogueira, R. et al. (2020). *Document Ranking with a Pretrained Sequence-to-Sequence Model.*
- **[Pyserini]**
  - Lin, J. et al. (2021). *Pyserini: A Python Toolkit for Reproducible Information Retrieval Research with Sparse and Dense Representations.*

## Reproducibility checklist

- **[Inputs]**
  - Queries: `Files-20260104/queriesROBUST.txt`
  - Judged qrels: `Files-20260104/qrels_50_Queries`
- **[Prebuilt indices]**
  - `robust04`
  - `beir-v1.0.0-robust04.splade-pp-ed`
  - `beir-v1.0.0-robust04.splade-v3`
  - `beir-v1.0.0-robust04.bge-base-en-v1.5.hnsw`
- **[Caching]**
  - Cache directory: `CACHE_DIR` (default `/workspace/.cache`)
- **[Strict rerun switches]**
  - Set `FORCE_REGEN_RUNS=True`
  - Set `FORCE_CACHE_REFRESH=True`
  - Optionally set `FORCE_REGEN_HYDE=True`
